---
Selection Pocking Behavior in videos for FERAL - First dataset Built
---

In [1]:
import os
import cv2
import pandas as pd
from pathlib import Path

---
Configuration
---

In [ ]:

# ------------ CONFIGURE HERE ------------
CSV_FOLDER = r"D:\MasterThesis\Data\Boris_annotation\Toni"   # folder with CSV
VIDEO_FOLDER = r"D:\MasterThesis\Data\videos\Orangutans\Camera_1"      # folder with MP4
OUTPUT_FOLDER = r"D:\MasterThesis\Data\poking\poking_clips_Toni"

TRACKING_CSV = os.path.join(OUTPUT_FOLDER, "poking_tracking_Toni2.csv")

#CSV_FILENAME = "Daza_4.8.22_Test3_T1_AE.csv"           # one CSV to start

COL_VIDEO = "Observation id"                # column containing video filename
COL_MODIFIER = "Modifier #1"        # column with behavior labels
COL_IMG_START = "Image index start"  # starting frame index for poking

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
# ---------------------------------------


---
Functions
---

In [3]:
def find_video_path(video_base: str) -> str:
    """
    Given the base name from "Observation id" column (e.g. "Daza_18.8.22_Test6_T1"),
    build candidate path with .MP4 extension and find the video in VIDEO_FOLDER.
    """
    # Construct expected filename by adding .MP4 extension (uppercase, as in your videos)
    video_name = video_base + ".MP4"                       # ← NEW: append .MP4 as you requested
    
    # Build full path using VIDEO_FOLDER + constructed name
    candidate = os.path.join(VIDEO_FOLDER, video_name)     # e.g. "D:\videos\Daza_18.8.22_Test6_T1.MP4"
    
    # Check if exact match exists
    if os.path.isfile(candidate):
        print(f"  ✅ Found video: {video_name}")            # Debug print for confirmation
        return candidate

    # Fallback: try case-insensitive match (e.g. if video is .mp4 lowercase)
    lower_target = video_name.lower()                      # "daza_18.8.22_test6_t1.mp4"
    for f in os.listdir(VIDEO_FOLDER):
        if f.lower() == lower_target:                      # Ignore case for matching
            full_path = os.path.join(VIDEO_FOLDER, f)
            print(f"  ✅ Found case-insensitive: {f}")
            return full_path

    # If no match, raise clear error with suggestions
    available = [f for f in os.listdir(VIDEO_FOLDER) if f.lower().startswith(video_base.lower())]
    suggestions = ", ".join(available[:5]) if available else "none"
    raise FileNotFoundError(f"Video not found for Observation id '{video_base}' → '{video_name}'\n"
                            f"Available videos starting with '{video_base}': {suggestions}")


In [4]:
def get_video_props(path: str):
    """Open video and return (cap, fps, total_frames)."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    return cap, fps, total

In [5]:

def play_segment(cap, start_frame: int, end_frame: int, fps: float):
    """
    Play segment from start_frame to end_frame (inclusive) with given fps.
    Returns a key pressed by user: 'v','m','b','e','s','r','q'.
    """
    print(f"\n🎬 Playing frames {start_frame} → {end_frame} ({end_frame-start_frame+1} frames)")
    print("Commands: V=Validate&Save | M=Manual start/end | B=Adjust start | E=Adjust end | R=Restart | H=Hide | Q=Quit")
    
    # Reset to start frame each play
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)          # Jump to beginning of segment

    delay = int(1000 / fps) if fps > 0 else 33             # ms per frame (explained previously)

    while True:
        current = int(cap.get(cv2.CAP_PROP_POS_FRAMES))    # Get current frame number
        if current > end_frame:
            break                                          # End of segment reached

        ret, frame = cap.read()                            # Read next frame
        if not ret:
            break                                          # No more frames

        # Overlay with info (frame numbers + commands)
        txt = f"[{start_frame}-{end_frame}] current {current} | V/M/B/E/R/H/Q"
        cv2.putText(frame, txt, (30, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        cv2.imshow("Poking validation", frame)
        key = cv2.waitKey(delay) & 0xFF                    # Wait frame time, capture key [web:35]

        # Check for action keys DURING playback
        if key in [ord('v'), ord('V'), ord('m'), ord('M'), ord('b'), ord('B'),
                   ord('e'), ord('E'), ord('s'), ord('S'), ord('r'), ord('R'), ord('h'), ord('H'), ord('q'), ord('Q')]:
            return chr(key).lower()                        # Return lowercase key
    # After full playthrough, wait for decision
    print("\n⏹️  End of segment reached. Choose command above.")
    while True:
        key = cv2.waitKey(0) & 0xFF                       # Wait indefinitely for key
        if key in [ord('v'), ord('V'), ord('m'), ord('M'), ord('b'), ord('B'),
                   ord('e'), ord('E'), ord('s'), ord('S'), ord('r'), ord('R'), ord('h'), ord('H'), ord('q'), ord('Q')]:
            return chr(key).lower()


In [6]:
def cut_and_save_segment(input_path: str, start_frame: int, end_frame: int,
                         fps: float, output_path: str):
    """
    Cut [start_frame, end_frame] from input_path and save to output_path.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video for cutting: {input_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frame_count = 0
    while True:
        current = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        if current > end_frame:
            break

        ret, frame = cap.read()
        if not ret:
            break


        out.write(frame)
        frame_count += 1 

    cap.release()
    out.release()
    print(f"📹 Extracted {frame_count} clean frames to {output_path}")


In [7]:
def main():
    total_poking_count = 0
    csv_files = sorted([p for p in Path(CSV_FOLDER).iterdir()
                       if p.suffix.lower() == '.csv'])
    
    print(f"📁 Found {len(csv_files)} CSV files in {CSV_FOLDER}")
    for csv_file in csv_files:
        df_temp = pd.read_csv(str(csv_file))
        poking_in_csv = df_temp[df_temp[COL_MODIFIER] == "poking"]
        total_poking_count += len(poking_in_csv)
        print(f"   {csv_file.name}: {len(poking_in_csv)} poking rows")
    
    print(f"\n🎯 TOTAL: {total_poking_count} behavioral clips to validate")
    print(f"{'='*70}\n")
    
    if total_poking_count == 0:
        print("❌ No poking clips found anywhere!")
        return

    cv2.namedWindow("Poking validation", cv2.WINDOW_NORMAL)
    tracking_records = []
    processed_count = 0

    for csv_file in csv_files:
        csv_path = str(csv_file)
        csv_filename = os.path.basename(csv_path)
        print(f"\n{'='*60}")
        print(f"📄 Processing CSV: {csv_file.name}")
        print(f"{'='*60}")
        
        df = pd.read_csv(csv_path)
        df_poking = df[df[COL_MODIFIER] == "poking"]

        if df_poking.empty:
            print(f"   No poking rows → skipping")
            continue

        for idx, row in df_poking.iterrows():
            processed_count += 1

            # FIRST: load video + define variables
            video_name = row[COL_VIDEO]
            original_start = int(row[COL_IMG_START])

            try:
                video_path = find_video_path(video_name)
            except FileNotFoundError as e:
                    print(f"❌ Video not found for {video_name} in {csv_filename}")
                    print(f"   → Skipping this annotation.")

                    record = {
                        'video_name': video_name,
                        'csv_file': csv_filename,
                        'behavior': 'poking_VIDEO_NOT_FOUND',
                        'original_img_start': original_start,
                        'final_img_start': -1,
                        'final_img_stop': -1,
                        'start_frame_diff': 0,
                        'end_frame_diff': 0,
                        'behavior_frames': 0,
                        'clip_filename': '',
                        'remarks': 'video_not_found'
                    }

                    tracking_records.append(record)

                    # Save CSV immediately so missing videos are recorded
                    df_track = pd.DataFrame(tracking_records)
                    df_track.to_csv(TRACKING_CSV, index=False)

                    continue
            
            cap, fps, total_frames = get_video_props(video_path)

            start_frame = max(0, original_start -25)
            end_frame = min(total_frames - 1, original_start + 35)

            # Then compute remaining and print header ONCE
            remaining = total_poking_count - processed_count + 1

            print(f"\n[{processed_count}/{total_poking_count}] 🎥 {video_name} | original frame {original_start}")
            print(f"   ⏳ Remaining to review: {remaining} clips")
            print(f"   | working frames {start_frame} → {end_frame}")
            print(f"{'='*60}")

            while True:
                print(f"🔄 [{video_name}] frames {start_frame} → {end_frame} (orig: {original_start})")
                print("V=Validate&Save | M=Manual | B=Start+/- | E=End+/- | R=Replay | H=Hide/Skip | Q=Quit")
                
                key = play_segment(cap, start_frame, end_frame, fps)
                print(f"→ User pressed: '{key}'")

                if key == 'q':
                    print(f"👋 QUIT at #{processed_count} | Remaining: {total_poking_count - processed_count}")
                    cap.release()
                    break

                elif key == 'v':
                    print(f"✅ VALIDATED #{processed_count} | Remaining: {total_poking_count - processed_count}")
                    base = os.path.splitext(os.path.basename(video_path))[0]
                    duration_frames = end_frame - start_frame + 1
                    out_name = f"{base}_poking_{start_frame:06d}_{end_frame:06d}_{duration_frames:03d}f.mp4"
                    out_path = os.path.join(OUTPUT_FOLDER, out_name)
                    
                    record = {
                        'video_name': base,
                        'csv_file': csv_filename,
                        'behavior': 'poking',
                        'original_img_start': original_start,
                        'final_img_start': start_frame,
                        'final_img_stop': end_frame,
                        'start_frame_diff': start_frame - original_start,
                        'end_frame_diff': end_frame - original_start, 
                        'behavior_frames': duration_frames,
                        'clip_filename': out_name
                    }
                    tracking_records.append(record)
                    
                    print(f"💾 Saving '{out_name}' ({duration_frames} frames)")
                    cut_and_save_segment(video_path, start_frame, end_frame, fps, out_path)
                    print(f"✅ Saved: {out_path}")
                    
                    df_track = pd.DataFrame(tracking_records)
                    df_track.to_csv(TRACKING_CSV, index=False)
                    print(f"📊 Tracking updated: {len(tracking_records)} records")
                    
                    cap.release()
                    break

                elif key == 'h':
                    remarks = input("Remarks (or Enter for none): ").strip() or "bad_quality"
                    print(f"⏭️  HIDDEN #{processed_count} | Remaining: {total_poking_count - processed_count}")

                    base = os.path.splitext(os.path.basename(video_path))[0]
                    record = {
                        'video_name': base,
                        'csv_file': csv_filename,
                        'behavior': 'poking_HIDDEN',
                        'original_img_start': original_start,
                        'final_img_start': -1,
                        'final_img_stop': -1,
                        'start_frame_diff': 0,
                        'end_frame_diff': 0,
                        'behavior_frames': 0,
                        'clip_filename': '',
                        'remarks': remarks
                    }
                    tracking_records.append(record)
                    
                    df_track = pd.DataFrame(tracking_records)
                    df_track.to_csv(TRACKING_CSV, index=False)
                    print(f"📊 Logged HIDDEN clip: '{remarks}' | Total: {len(tracking_records)} records")
                    
                    cap.release()
                    break

                elif key == 'r':
                    print("🔄 Replaying current segment...")
                    continue

                elif key == 'm':
                    print("\n📝 MANUAL mode")
                    try:
                        new_start = int(input(f"New START frame (0–{total_frames-1}): "))
                        new_end = int(input(f"New END frame ({new_start}–{total_frames-1}): "))
                        new_start = max(0, min(new_start, total_frames - 1))
                        new_end = max(new_start, min(new_end, total_frames - 1))
                        start_frame, end_frame = new_start, new_end
                        print(f"✅ Updated: {start_frame} → {end_frame}")
                    except ValueError:
                        print("❌ Invalid numbers, keeping previous.")
                    continue

                elif key == 'b':
                    print("\n⬅️  Adjust START")
                    try:
                        offset = int(input("Offset (+/- frames): "))
                        new_start = max(0, min(start_frame + offset, end_frame))
                        start_frame = new_start
                        print(f"✅ New start: {start_frame}")
                    except ValueError:
                        print("❌ Invalid offset.")
                    continue

                elif key == 'e':
                    print("\n➡️  Adjust END")
                    try:
                        offset = int(input("Offset (+/- frames): "))
                        new_end = max(start_frame, min(end_frame + offset, total_frames - 1))
                        end_frame = new_end
                        print(f"✅ New end: {end_frame}")
                    except ValueError:
                        print("❌ Invalid offset.")
                    continue
            
            if 'cap' in locals() and cap.isOpened():
                cap.release()

    cv2.destroyAllWindows()

    # Optional final summary
    validated = sum(1 for r in tracking_records if r['behavior'] == 'poking')
    hidden = sum(1 for r in tracking_records if 'HIDDEN' in r['behavior'])
    print(f"\n🎉 FINISHED!")
    print(f"   ✅ Validated: {validated} clips")
    print(f"   ⏭️  Hidden: {hidden} clips")
    print(f"   📊 Total tracked: {len(tracking_records)} | {TRACKING_CSV}")


---
START here
---

In [9]:
if __name__ == "__main__":
    main()

📁 Found 9 CSV files in D:\MasterThesis\Data\Boris_annotation\Toni
   PST_Toni_olddate_hab2_AE.csv: 0 poking rows
   Toni_HL_11.3.23_Test4_T1_AE.csv: 0 poking rows
   Toni_HL_17.3.23_Test5_T1_AE.csv: 14 poking rows
   Toni_HL_7.3.23_Test3_T1_AE.csv: 34 poking rows
   Toni_HL_olddate_Test1_T1_AE.csv: 2 poking rows
   Toni_HL_oldddate_Test2_T1_AE.csv: 12 poking rows
   Toni_HL_Test6_18.3.23_T1_AE.csv: 2 poking rows
   Toni_HL_Test7_19.3.23_T1_AE.csv: 20 poking rows
   Toni_HL_Test7_19.3.23_T2_AE.csv: 8 poking rows

🎯 TOTAL: 92 behavioral clips to validate


📄 Processing CSV: PST_Toni_olddate_hab2_AE.csv
   No poking rows → skipping

📄 Processing CSV: Toni_HL_11.3.23_Test4_T1_AE.csv
   No poking rows → skipping

📄 Processing CSV: Toni_HL_17.3.23_Test5_T1_AE.csv
  ✅ Found video: Toni_HL_17.3.23_Test5_T1.MP4

[1/92] 🎥 Toni_HL_17.3.23_Test5_T1 | original frame 2494
   ⏳ Remaining to review: 92 clips
   | working frames 2469 → 2529
🔄 [Toni_HL_17.3.23_Test5_T1] frames 2469 → 2529 (orig: 2494)
V